# Submission WSense — pipeline ibrida

**Dataset da allegare:** la competizione + il dataset con `hybrid_codec.py` + il dataset
con `wsense_restorer.pth` e `wsense_operating_point.json` prodotti dal notebook di training.

Compila `TEAM`, poi **Run All**. Se il restauratore non viene trovato, il notebook lavora
comunque con il solo backend classico (qualita' inferiore ma submission valida) invece di
fermarsi.

## 0. Setup

In [ ]:
import sys, glob, pathlib, os, json, time
try:
    from challenge_common import *
except ImportError:
    found = next((p for pat in ["/kaggle/input/*/challenge_common.py", "/kaggle/input/*/*/challenge_common.py",
                  "/kaggle/input/*/*/*/challenge_common.py", "challenge_common.py"]
                  for p in glob.glob(pat)), None)
    if found is None:
        raise FileNotFoundError("challenge_common.py non trovato")
    sys.path.insert(0, str(pathlib.Path(found).parent))
    from challenge_common import *
import numpy as np, pandas as pd, torch

_own = next((pathlib.Path(p).parent for pat in ["/kaggle/input/*/hybrid_codec.py",
             "/kaggle/input/*/*/hybrid_codec.py", "/kaggle/input/*/*/*/hybrid_codec.py",
             "hybrid_codec.py"] for p in glob.glob(pat)), None)
if _own is None:
    raise FileNotFoundError("hybrid_codec.py non trovato - allega il dataset")
sys.path.insert(0, str(_own))

# Percorsi espliciti della fase di test privata.
DATA_ROOT_EXPLICIT = "/kaggle/input/competitions/wsense-training-camp/underwater-compression_v2_test/underwater-compression_v2_test"
SAMPLE_EXPLICIT    = "/kaggle/input/competitions/wsense-training-camp/sample_submission_with_private.csv"

def _find_real_data_root():
    """Rete di sicurezza: se il percorso esplicito non esiste (rinominato, fase diversa),
    cerca la cartella che contiene task1/task2/task3 a qualunque profondita'. Senza questo,
    configure() ripiegherebbe in silenzio su dati SINTETICI e la submission sarebbe
    costruita su rumore generato, con un fallimento visibile solo a punteggio ricevuto."""
    for root, dirs, files in os.walk("/kaggle/input"):
        if all(t in dirs for t in ("task1", "task2", "task3")):
            return root
    return None

if os.path.isdir(DATA_ROOT_EXPLICIT):
    _dr = DATA_ROOT_EXPLICIT
else:
    print("percorso esplicito dei dati non trovato, cerco automaticamente...")
    _dr = _find_real_data_root()
    if not _dr:
        raise FileNotFoundError("task1/task2/task3 non trovati sotto /kaggle/input")
print("dati:", _dr)
configure(data_root=_dr)

if os.path.exists(SAMPLE_EXPLICIT):
    SAMPLE = SAMPLE_EXPLICIT
else:
    print("sample submission esplicita non trovata, cerco automaticamente...")
    SAMPLE = next((p for pat in ["/kaggle/input/*/sample_submission*.csv",
                                 "/kaggle/input/*/*/sample_submission*.csv",
                                 "/kaggle/input/*/*/*/sample_submission*.csv"]
                   for p in sorted(glob.glob(pat))), "sample_submission.csv")
print("sample submission:", SAMPLE)
CKPT = next((p for pat in ["/kaggle/input/*/wsense_restorer.pth",
             "/kaggle/input/*/*/wsense_restorer.pth", "/kaggle/input/*/*/*/wsense_restorer.pth",
             "wsense_restorer.pth"] for p in glob.glob(pat)), None)
OP_PATH = next((p for pat in ["/kaggle/input/*/wsense_operating_point.json",
                "/kaggle/input/*/*/wsense_operating_point.json",
                "/kaggle/input/*/*/*/wsense_operating_point.json",
                "wsense_operating_point.json"] for p in glob.glob(pat)), None)

TEAM = "TEAM NAME"          # <-- COMPILA
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import hybrid_codec as HC
from hybrid_codec import (HybridCodec, HybridBudgetCodec, HybridGroupCodec,
                          task_score, EQUAL_WEIGHTS, TASK2_MAX_BYTES, BACKEND)

if CKPT is None:
    print("ATTENZIONE: wsense_restorer.pth non trovato -> solo backend classico, "
          "niente restauro. La submission resta valida ma la qualita' e' inferiore.")
OP = json.load(open(OP_PATH)) if OP_PATH else {
    "task1_scale": 0.35, "task1_quality": 50, "task3_scale": 0.35, "task3_quality": 50}
print("backend:", BACKEND, "| device:", DEVICE)
print("punto operativo:", json.dumps(OP))

## 1. Codec

In [ ]:
kw = dict(weights_path=CKPT, device=DEVICE, use_restorer=CKPT is not None)
codec        = HybridCodec(quality=OP["task1_quality"], scale=OP["task1_scale"], **kw)
budget_codec = HybridBudgetCodec(**kw)
group_codec  = HybridGroupCodec(quality=OP["task3_quality"], scale=OP["task3_scale"], **kw)
print(f"task1: scale={codec.scale} q={codec.quality}")
print(f"task3: scale={group_codec.scale} q={group_codec.quality}")
print(f"task2: cap {TASK2_MAX_BYTES} B (oltre, il punteggio del task e' dimezzato)")

## 2. Controllo di validita'

In [ ]:
FULL_SELFCHECK = False

# Il dataset di test contiene solo 'public_test': niente 'val'. Usiamo il primo split
# disponibile invece di darlo per scontato — il controllo serve a verificare il contratto
# (determinismo, shape, budget), non a misurare qualità su dati di sviluppo.
import challenge_common as _cc
_idx = _cc.index_images(_cc.task_root("task1"), _cc.TASK_CFG["task1"]["splits"])
_splits = sorted(_idx.split.unique())
_SPLIT = "val" if "val" in _splits else _splits[0]
print("split usato per il controllo:", _SPLIT, "| disponibili:", _splits)

_p = load_split("task1", _SPLIT, n_images=1, verbose=False).path.iloc[0]
_im = load_image(_p)
_a, _b = bytes(codec.encode(_im)), bytes(codec.encode(_im))
assert _a == _b, "encode() non deterministico"
_r = codec.decode(_a)
assert _r.shape == _im.shape and _r.dtype == np.uint8, f"shape {_r.shape}"
_P, _M, _CR = psnr(_im, _r), ms_ssim(_im, _r), compression_ratio(_im, _a)
print(f"  task1: {len(_a)} B  CR={_CR:.1f}  PSNR={_P:.2f}dB  MS-SSIM={_M:.4f}  "
      f"S={task_score(_P,_M,_CR,'task1'):.4f}")

_buds = sorted(set(scored_items(SAMPLE, "task2").values())) or BUDGETS
_p2 = load_split("task2", _SPLIT, n_images=1, verbose=False).path.iloc[0]
_im2 = load_image(_p2)
for _bd in _buds:
    _pay = bytes(budget_codec.encode_at(_im2, _bd))
    _r2 = budget_codec.decode(_pay)
    _lim = min(_bd, TASK2_MAX_BYTES)
    assert len(_pay) <= _lim, f"fuori budget: {len(_pay)} > {_lim}"
    assert _r2.shape == _im2.shape
    print(f"  task2 budget {_bd} -> {len(_pay)} B (cap {_lim}, {100*len(_pay)/_lim:.0f}% usato)  "
          f"PSNR={psnr(_im2,_r2):.2f}dB")

_g, _ = load_groups("task3", _SPLIT, n_scenes=1, verbose=False)
_paths = list(_g.values())[0][:4]
_imgs = [load_image(p) for p in _paths]
_pays = [bytes(x) for x in group_codec.encode_group(_imgs)]
_recs = group_codec.decode_group(_pays)
for _x, _y in zip(_imgs, _recs):
    assert _y.shape == _x.shape and _y.dtype == np.uint8
_cr3 = sum(image_bytes(i) for i in _imgs) / sum(len(p) for p in _pays)
print(f"  task3: CR={_cr3:.1f}  PSNR={np.mean([psnr(a,b) for a,b in zip(_imgs,_recs)]):.2f}dB")
print("\ncontrollo veloce: PASSATO")

## 3. `submission.csv`

In [ ]:
t0 = time.time()
path = make_submission(codec=codec, budget_codec=budget_codec, group_codec=group_codec,
                       sample_submission=SAMPLE)
sub = pd.read_csv(path); samp = pd.read_csv(SAMPLE)
assert sub.isna().sum().sum() == 0, "il CSV contiene null: Kaggle lo rifiuterebbe"
assert list(sub.columns) == list(samp.columns), "colonne diverse da sample_submission"
assert len(sub) == len(samp), "numero di righe diverso da sample_submission"
print(f"\nteam={TEAM}  |  {(time.time()-t0)/60:.1f} min  |  {sub.shape}  |  pronto per Submit")